# Examples of how to use the code and to create Composite profiles (e.g. stars + DM)

In [ ]:
import adiabatic_tides as at
import numpy as np
import matplotlib.pyplot as plt

### A single profile and some properties
* units are:
* length: kpc
* mass: Msol
* velocity km/s
(note that this is different from the original repository where default length was Mpc)

anisotropy may be in range (-0.5,0.)

In [ ]:
nfw = at.profiles.NFWProfile(conc=12., m200c=1e10, anisotropy=0.0)
print(nfw)
ri = np.logspace(-6,4)
plt.loglog(ri, nfw.vcirc(ri))
rmax, vmax = nfw.rmax_vmax()
plt.axvline(rmax)
plt.axhline(vmax)

### Tidal iterations

In [ ]:
# for a nfw profile the natural scale of the tidal field is given by lambdas = |acc(rs)/rs|
lambdas = np.abs(nfw.accr(nfw.rs)/nfw.rs)
atr = at.adiabatic.AdiabaticTidalTransformation(nfw, tide=1e-2*lambdas) # Create an object that defines the transformation
atr.run() # Will run the iterations until converged

# Plot the iteration history, usually only numerically interesting
for ind in range(len(atr.history)):
    # Assemble the profile at iteration ind
    # Can be used mostly the same way as the initial profile
    new_prof = atr.assemble_total_profile(iter=ind) 
    # One important difference is that it includes the external contribution from the tidal field
    # (Corresponding to a constant negative density.) 
    # In almost all cases, you want to use only the internal component, that you can access by mode="self"
    ri = np.logspace(-1,2)
    plt.loglog(ri, new_prof.density(ri, mode="self"), label='i=%d' % ind)
plt.xlim(1e0,1e2)
plt.ylim(1e2,1e8)

plt.plot(ri, nfw.density(ri), label='initial', ls="dashed", color="black")

# Assemble the final profile, which is the last iteration
final_prof = atr.assemble_total_profile()

plt.plot(ri, final_prof.density(ri, mode="self"), label='final', ls="dotted", color="black")
plt.legend()

## Composite profile

In [ ]:
# E.g. create an analytic profile
nfw = at.profiles.NFWProfile(conc=12., m200c=1e10)
print(nfw.r200c, nfw.rs) # in kpc

# E.g. create a numerical profile for the stars
# Note that we choose one with artificially large mass in the stars
# to create an interesting case where the self-gravity of both components matters
def rho(x):
    return (x + nfw.rs)**-4 * 5e3*nfw.rhoc

# When providing radii, make sure that you resolve some regime below and above the region of interest
# Also more points are more accurate, but typically ~ 1000 are well more than enough
ri = np.logspace(-6,4, 1000)
starprof = at.profiles.NumericalProfile(ri, rho(ri))

# You can use any name of your liking for different components and set any number of components
initial_prof = at.profiles.CompositeProfile(dm=nfw, stars=starprof) 

In [ ]:
# individual components can be accessed by adding mode=...
plt.loglog(ri, initial_prof.density(ri, mode="dm"), label='dm')
plt.loglog(ri, initial_prof.density(ri, mode="stars"), label='stars')
plt.loglog(ri, initial_prof.density(ri), label='self', ls="dashed", color="black")
plt.legend()
plt.ylim(1e-3,1e13)

In [ ]:
# Create an adiabatic transformation
final_prof = at.adiabatic.AdiabaticTidalTransformation(initial_prof, tide=1e-2*np.abs(nfw.accr(nfw.rs)/nfw.rs)).run(eps=1e-3).assemble_total_profile()

In [ ]:
ri = np.logspace(-1,3, 400)
plt.loglog(ri, initial_prof.density(ri, mode="dm"), label='dm initial')
plt.loglog(ri, initial_prof.density(ri, mode="stars"), label='stars initial')
plt.loglog(ri, initial_prof.density(ri), label="total initial", color="black", ls="dotted")

plt.loglog(ri, final_prof.density(ri, mode="stars"), label='stars final', ls="dashed")
plt.loglog(ri, final_prof.density(ri, mode="dm"), label='dm final', ls="dashed", color="black")
plt.loglog(ri, final_prof.density(ri, mode="self"), label='total final', ls="dotted", color="grey")
plt.xlabel("r [kpc]")
plt.ylabel(r"$\rho$ [M$_\odot$ kpc$^{-3}$]")

plt.axvline(final_prof.rtid(), label="tidal radius")
plt.ylim(1e2,1e9)
plt.xlim(0.1,4e2)
plt.legend()

In [ ]:
# If you are interested in summary statistics of the profile, you can get them easily, e.g.
# However, watch out that some may slightly depend on whether you include the external contribution or not
print(final_prof.rmax_vmax(mode="self"))
print(final_prof.rmax_vmax(mode="total"))
print(final_prof.rmax_vmax(mode="stars"))
print(final_prof.rmax_vmax(mode="dm"))

print(final_prof)
print(final_prof.profiles["tide"]) # This is the external contribution